In [34]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

import re 
from bs4 import BeautifulSoup

import emoji

Loads Twitter dataset with sentiment labels (Positive, Negative, Neutral, Irrelevant)

In [35]:
# loading the dataset

file_path = r"./twitter_training.csv"
df = pd.read_csv(file_path , header = None ,names=['number' , 'Border' , 'label' , 'message']) # Adjusting the column names

In [36]:
df.drop(['Border' , 'number'] , axis=1 , inplace = True)

Loading & Cleaning: Drops unnecessary columns and null values

In [37]:
df.dropna(inplace = True)

In [38]:
df['message'] = df['message'].str.lower()

df.head()

,label,message
0,Positive,im getting on borderlands and i will murder yo...
1,Positive,i am coming to the borders and i will kill you...
2,Positive,im getting on borderlands and i will kill you ...
3,Positive,im coming on borderlands and i will murder you...
4,Positive,im getting on borderlands 2 and i will murder ...


Text Preprocessing:

Lowercasing

HTML tag removal (BeautifulSoup)

URL removal

Punctuation removal

Emoji conversion to text (using emoji library)

In [39]:
from bs4 import BeautifulSoup

def remove_html(text):

    clean_text = BeautifulSoup(text , 'html.parser')

    return clean_text.get_text()

In [40]:
df['message'] = df['message'].apply(remove_html)

display(df['message'].head(2))

C:\Users\Pc\AppData\Local\Temp\ipykernel_21864\781006695.py:5: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a filename than HTML or XML.

If you meant to use Beautiful Soup to parse the contents of a file on disk, then something has gone wrong. You should open the file first, using code like this:

    filehandle = open(your filename)

You can then feed the open filehandle into Beautiful Soup instead of using the filename.

However, if you want to parse some data that happens to look like a filename, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  clean_text = BeautifulSoup(text , 'html.parser')


0    im getting on borderlands and i will murder yo...
1    i am coming to the borders and i will kill you...
Name: message, dtype: object

In [41]:
import re

def clean_url(text):
    
    return re.sub(r'http\S+|www\S+', '', text)\


df['message'] = df['message'].apply(clean_url)

In [42]:
def remove_punctuation(text):
    
    return re.sub(r'[^\w\s]', '', text)
df['message'] = df['message'].apply(remove_punctuation)

In [43]:

# در اینجا ایموجی ها را تبدیل به کلمه می کنیم 
def remove_emojis(text):
    return emoji.demojize(text)

df['message'] = df['message'].apply(remove_emojis)

In [44]:
def clean_text(text):
    if not isinstance(text, str):
        return text
    
    text = text.lower() 
    text = remove_html(text) 
    text = clean_url(text)  
    text = remove_punctuation(text) 

    text = remove_emojis(text) 
    
    return text

In [45]:
# let's give it a test !

new_text = "Heyyyy!!! 😊 Check this out: https://example.com <b>Awesome!</b>"
cleaned_text = clean_text(new_text)
print(cleaned_text)

heyyyy  check this out  awesome


In [46]:
df

,label,message
0,Positive,im getting on borderlands and i will murder yo...
1,Positive,i am coming to the borders and i will kill you...
2,Positive,im getting on borderlands and i will kill you all
3,Positive,im coming on borderlands and i will murder you...
4,Positive,im getting on borderlands 2 and i will murder ...
...,...,...
74677,Positive,just realized that the windows partition of my...
74678,Positive,just realized that my mac window partition is ...
74679,Positive,just realized the windows partition of my mac ...
74680,Positive,just realized between the windows partition of...


In [ ]:
#Neutral and Irrelevant both mapped to class 2 (could cause confusion)
df['label'] = df['label'].map({'Positive' : 1 ,  'Negative' : 0 ,'Neutral':2 , 'Irrelevant' : 2 })

In [ ]:
from sklearn.model_selection import train_test_split

X = df['message']
y = df['label'] 

X_train1 , X_test , y_train1 , y_test = train_test_split(X,y , random_state = 42 , test_size = 0.2  , shuffle = True)
X_train , X_val , y_train , y_val = train_test_split(X_train1 , y_train1 , random_state = 42 , test_size = 0.15  , shuffle = True)

In [ ]:
import tensorflow as tf 

Tokenization: Converts text to sequences using Keras Tokenizer

In [50]:
from tensorflow.keras.preprocessing.text import Tokenizer


tokenizer = Tokenizer(oov_token='<OOV>') 
tokenizer.fit_on_texts(X_train)

vocab_size = len(tokenizer.word_index) + 1
print(f"Vocab size: {vocab_size}")

Vocab size: 34824


In [51]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_val_seq = tokenizer.texts_to_sequences(X_val)

In [52]:

max_len = max(len(tokens) for tokens in X_train_seq)
print("Maximum sequence length (maxlen):", max_len)


Maximum sequence length (maxlen): 99


Padding: Makes all sequences equal length (max_len=99)

In [53]:
from tensorflow.keras.preprocessing.sequence import pad_sequences



X_train_padded = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_padded = pad_sequences(X_test_seq, maxlen=max_len, padding='post')
X_val_padded = pad_sequences(X_val_seq, maxlen=max_len, padding='post')

In [54]:
# Define vocab size based on the tokenizer
vocab_size = len(tokenizer.word_index) + 1

print(vocab_size)

34824


In [ ]:

model = tf.keras.models.Sequential([

    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=100),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128 , return_sequences = True , dropout = 0.2 , recurrent_dropout = 0.2)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64 , dropout = 0.2 , recurrent_dropout = 0.2)),
    tf.keras.layers.Dense(64 , activation='relu'  , kernel_initializer = 'he_normal'),
    tf.keras.layers.Dense(3 , activation = 'softmax')

])

In [ ]:

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

In [57]:

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy"]
)

In [ ]:

#history = model.fit(X_train_padded,  y_train,  validation_data=(X_val_padded, y_val),  # Validation setbatch_size=32,  epochs=30,  callbacks=[early_stopping , reduce_lr],  # to Prevent overfittingverbose=1 )

In [59]:
# ددر این قسمت یک خلاصه از مدل می گیریم و می بیینم که چند میلیون پارامتر دارد
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, None, 100)         3482400   
                                                                 
 bidirectional_2 (Bidirecti  (None, None, 256)         234496    
 onal)                                                           
                                                                 
 bidirectional_3 (Bidirecti  (None, 128)               164352    
 onal)                                                           
                                                                 
 dense_2 (Dense)             (None, 64)                8256      
                                                                 
 dense_3 (Dense)             (None, 3)                 195       
                                                                 
Total params: 3889699 (14.84 MB)
Trainable params: 388

In [ ]:
import pickle
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# Load the saved model
loaded_model = tf.keras.models.load_model('LSTM_Sentiment_analysis.h5')
y_probs = loaded_model.predict(X_test_padded)
y_pred = np.argmax(y_probs, axis=1)

463/463 [==============================] - 15s 31ms/step


In [61]:

loss, accuracy = loaded_model.evaluate(X_test_padded, y_test)
print(f"Loss: {loss}, Accuracy: {accuracy}")

463/463 [==============================] - 16s 34ms/step - loss: 0.3217 - accuracy: 0.8836
Loss: 0.32172319293022156, Accuracy: 0.8835811018943787


## Confusion Matrix

In [62]:
from sklearn.metrics import classification_report
report = classification_report(y_test, y_pred)
# دقت مدل را روی داده تست می سنجیم
# the report
print("Classification Report:")
print(report)

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.88      0.88      4380
           1       0.88      0.87      0.87      4119
           2       0.89      0.90      0.89      6301

    accuracy                           0.88     14800
   macro avg       0.88      0.88      0.88     14800
weighted avg       0.88      0.88      0.88     14800



In [ ]:

#model.save("LSTM_Sentiment_analysis.h5")

In [64]:
def preprocess_text(texts, tokenizer):

    text_seq = tokenizer.texts_to_sequences(texts)
    
    text_padded = pad_sequences(text_seq, maxlen=max_len, padding="post")
    
    return text_padded

In [65]:
def Predict(text , model , tokenizer):
    text = clean_text(text)
    
    text = [text]
    text = clean_text(text)
    text_padded =  preprocess_text(text , tokenizer)

    y_prob = model.predict(text_padded)
   
    y_pred = np.argmax(y_prob, axis=1)

    classes = ['Negative' , 'Positive' , 'Neutral']

    pred_class = classes[y_pred[0]]  # Get predicted class label
    pred_prob = y_prob[0][y_pred[0]] # get predicted prob


    return pred_class, pred_prob

In [67]:
new_text = "I'm HAPPY"

pred_class  , prob  = Predict(new_text , loaded_model , tokenizer)
print(f"Class Prediction is : {pred_class} with Probabilty {prob}")

1/1 [==============================] - 0s 29ms/step
Class Prediction is : Positive with Probabilty 0.7887118458747864
